In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
TAG = 'arw4'
MIN_PROB = 1e-12

In [4]:
pairs = pd.read_csv(INPUT / 'transitions.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [5]:
def trunclaplace_negloglike(alpha, values, modes):
    if not np.isfinite(alpha) or alpha <= 0:
        return np.inf
    values = np.asarray(values, dtype=float)
    modes = np.asarray(modes, dtype=float)
    log_density = -np.log(2 * alpha) - np.abs(values - modes) / alpha
    survival = 1 - stats.laplace.cdf(0, loc=modes, scale=alpha)
    survival = np.clip(survival, MIN_PROB, 1)
    return -np.sum(log_density - np.log(survival))
def fit_trunc_laplace(values, modes):
    values = np.asarray(values, dtype=float)
    modes = np.asarray(modes, dtype=float)
    scale_hi = max(10.0, 5 * np.std(values), 5 * np.mean(values), 500.0)
    result = minimize_scalar(trunclaplace_negloglike, args=(values, modes), bounds=(1e-08, scale_hi), method='bounded')
    return float(result.x)
def fit_arw4_block(subset, prior_beta=None):
    x = subset['q'].to_numpy(dtype=float)
    y = subset['q_next'].to_numpy(dtype=float)
    if prior_beta is None:
        denominator = max(np.median(x), EPS)
        prior_beta = float(np.clip(np.median(y) / denominator, 1e-06, 5.0))
    alpha0 = fit_trunc_laplace(y, prior_beta * x)
    result = minimize_scalar(lambda beta: trunclaplace_negloglike(alpha0, y, beta * x), bounds=(1e-06, 5.0), method='bounded')
    beta = float(result.x)
    alpha = fit_trunc_laplace(y, beta * x)
    return {'n': len(x), 'alpha': alpha, 'mode_beta': beta, 'nll': trunclaplace_negloglike(alpha, y, beta * x), 'mean_q_now': x.mean(), 'mean_q_next': y.mean(), 'mean_raw_delta': np.mean(y - x), 'median_raw_delta': np.median(y - x)}

In [6]:
stages = [('0-4',0,4), ('5-7',5,7), ('8-13',8,13), ('14-19',14,19)]
global_fit = fit_arw4_block(pairs)
rows = []
for label, start, end in stages:
    block = pairs.loc[pairs.age.between(start, end)]
    rows.append({'stage':label, 'start':start, 'end':end, **fit_arw4_block(block, global_fit['mode_beta'])})
params = pd.DataFrame(rows)
params.to_csv(DEST / 'parameters.csv', index=False)
pd.DataFrame([{'scale':q0.mean(), 'n':len(q0)}]).to_csv(DEST / 'initialization.csv', index=False)
display(params)

,stage,start,end,n,alpha,mode_beta,nll,mean_q_now,mean_q_next,mean_raw_delta,median_raw_delta
0,0-4,0,4,9789,4.493032,0.794375,27068.423528,6.634466,7.235996,0.601531,0.000000
1,5-7,5,7,5140,4.116321,0.748148,13912.655107,7.388392,7.060129,-0.328263,-0.171862
2,8-13,8,13,7664,3.797657,0.748148,20025.757747,6.642415,6.473341,-0.169074,-0.092021
3,14-19,14,19,4440,3.608783,0.774203,11397.449624,6.533666,6.423050,-0.110616,-0.060075
